# Transformer from scratch: Colab runner
Runtime > Change runtime type > **T4 GPU** (free). Data and results are saved to your Google Drive under `MyDrive/transformer/`, so a disconnect loses at most the run in progress. Re-running a cell skips runs that already finished.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/transformer'
!mkdir -p {DRIVE}/data {DRIVE}/runs
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!rm -rf /content/repo && git clone -q https://github.com/BhigaBilla/transformer_from_scratch /content/repo
%cd /content/repo
!pip -q install tokenizers datasets

## 1. Data (once, ~10-15 min): TinyStories download, BPE tokenizer, token files saved to Drive

In [ ]:
import os
if not os.path.exists(f'{DRIVE}/data/val.bin'):
    !python run.py prepare --data {DRIVE}/data
# local copy: reading from local disk is much faster than from Drive
!mkdir -p /content/data && cp {DRIVE}/data/* /content/data/ && ls -la /content/data

## 2. Runs
Groups: `baseline`, `pos`, `norm`, `heads`, `shape`, `extra`. Each run is ~41M tokens. Start with the baseline, seed 0.

In [ ]:
GROUPS = 'baseline'
SEEDS = '0'
!python run.py train --data /content/data --out {DRIVE}/runs --group {GROUPS} --seeds {SEEDS}

## 3. Results and samples

In [ ]:
!python analyze.py --out {DRIVE}/runs
!cp results.md {DRIVE}/ && cp -r plots {DRIVE}/

In [ ]:
!python sample.py --run {DRIVE}/runs/baseline_s0 --data /content/data -n 3